In [35]:
from ingest import load_faq_data
documents = load_faq_data()

In [36]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

113

In [37]:
documents = documents_llm

In [71]:
documents[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [38]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [39]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [40]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [41]:
from dotenv import load_dotenv
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

environment = os.getenv("OPENAI_API_KEY")
openai_client = OpenAI()

In [73]:
import json

user_prompt = json.dumps(doc)
print(len(doc))

print(user_prompt)

5
{"id": "4b30b918bc", "course": "llm-zoomcamp", "section": "Workshop: Open-Source Data Ingestion (dlt)", "question": "Error: How to fix requests library only installs v2.28 instead of v2.32 required for lancedb?", "answer": "If you encounter a 401 Client Error, it may indicate the need to grant access to the key or that the key is incorrect.\n\nTo install the correct version directly from the source, use the following command:\n\n```bash\npip install \"requests @ https://github.com/psf/requests/archive/refs/tags/v2.32.3.zip\"\n```"}


In [43]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [44]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [45]:
result = response.output_parsed

print(result)

questions=['I just found this course late — can I still join in now, or is it too late?', 'Am I allowed to start the course after it’s already begun?', 'If I join the course now, can I still get a certificate somehow?', 'Is there a deadline for handing in the project if I want the certificate?', 'Do I need to submit my project before submissions close in order to receive the certificate?']


In [46]:
print(result.questions)

['I just found this course late — can I still join in now, or is it too late?', 'Am I allowed to start the course after it’s already begun?', 'If I join the course now, can I still get a certificate somehow?', 'Is there a deadline for handing in the project if I want the certificate?', 'Do I need to submit my project before submissions close in order to receive the certificate?']


In [47]:
from evaluation_utils import llm_structured

In [48]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['I just found this course — can I still join and follow along?', 'Am I allowed to start the course late, or is it already too far along?', 'If I join now, can I still get a certificate somehow?', 'What do I need to do to be eligible for the certificate if I’m starting late?', 'Is there a deadline for submitting the project so I can still receive the certificate?']


In [49]:
usage.input_tokens, usage.output_tokens

(207, 94)

In [50]:
from evaluation_utils import calc_price

In [51]:
cost = calc_price(usage)

cost

{'input_cost': 0.00015525, 'output_cost': 0.000423, 'total_cost': 0.00057825}

In [52]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'I just found this course — can I still join and follow along?',
  'document': '74eb249bbf'},
 {'question': 'Am I allowed to start the course late, or is it already too far along?',
  'document': '74eb249bbf'},
 {'question': 'If I join now, can I still get a certificate somehow?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to be eligible for the certificate if I’m starting late?',
  'document': '74eb249bbf'},
 {'question': 'Is there a deadline for submitting the project so I can still receive the certificate?',
  'document': '74eb249bbf'}]

In [53]:
from evaluation_utils import llm_structured_retry

In [54]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [55]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [56]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/113 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

565

In [ ]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.08787300000000006

In [ ]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.08787300000000006

In [ ]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [ ]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)

# Ground Truth

In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [3]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [4]:
q = ground_truth[0]
q

{'question': 'I just found this course—can I still join it, or is it too late already?',
 'document': '74eb249bbf'}

In [5]:
doc_id = q["document"]
results = text_search(query=q["question"])

In [6]:
for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

74eb249bbf == 74eb249bbf: True
0fab61eca2 == 74eb249bbf: False
a9353fadfe == 74eb249bbf: False
9f689c185f == 74eb249bbf: False
977bf7786c == 74eb249bbf: False


In [7]:
relevance = []

for d in results:
    relevance.append(int(d["id"] == doc_id))

relevance

[1, 0, 0, 0, 0]

In [8]:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [9]:
q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)

I just found this course—can I still join it, or is it too late already?


[1, 0, 0, 0, 0]

In [10]:
q = ground_truth[11]
print(q["question"])
compute_relevance_text(q)

Is the Zoom URL for the live sessions shared with students, or is it only for the teaching team?


[1, 0, 0, 0, 0]

In [11]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [12]:
ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)

  0%|          | 0/15 [00:00<?, ?it/s]

In [13]:
relevance_total_text

[[1, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [14]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [15]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [16]:
relevance_total = compute_relevance_total(ground_truth_sample, text_search)
relevance_total

  0%|          | 0/15 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [17]:
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/565 [00:00<?, ?it/s]

# Hit Rate

In [18]:
example = [
    [1, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 1, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
]

In [19]:
cnt = 0

for line in example:
    if 1 in line:
        cnt = cnt + 1

cnt

14

In [20]:
cnt / len(example)

0.9333333333333333

In [21]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [22]:
hit_rate(example)

0.9333333333333333

In [23]:
total_score = 0.0

for line in example:
    for rank in range(len(line)):
        if line[rank] == 1:
            total_score = total_score + 1 / (rank + 1)
            break

total_score

12.333333333333332

In [24]:
total_score / len(example)

0.8222222222222222

In [25]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [26]:
mrr(example)

0.8222222222222222

In [27]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [28]:
evaluate(
    ground_truth,
    text_search
)

  0%|          | 0/565 [00:00<?, ?it/s]

{'hit_rate': 0.8336283185840708, 'mrr': 0.7076991150442473}

In [29]:
def search_boost(query, question_boost):
    boost_dict = {"question": question_boost, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [30]:
for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )
    print(f"boost={boost}: {result}")

  0%|          | 0/565 [00:00<?, ?it/s]

boost=0.5: {'hit_rate': 0.8920353982300885, 'mrr': 0.7952212389380529}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=1.0: {'hit_rate': 0.8938053097345132, 'mrr': 0.791474926253687}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=3.0: {'hit_rate': 0.8336283185840708, 'mrr': 0.7076991150442473}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=5.0: {'hit_rate': 0.7964601769911505, 'mrr': 0.6833923303834807}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=10.0: {'hit_rate': 0.7592920353982301, 'mrr': 0.6508849557522123}


In [31]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [32]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(
                f"Evaluating question_boost={question_boost},"
                f" answer_boost={answer_boost},"
                f" section_boost={section_boost}..."
            )
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


...

  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


  0%|          | 0/565 [00:00<?, ?it/s]

In [33]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

,question,answer,section,hit_rate,mrr
35,5.0,10.0,0.5,0.968142,0.871003
3,1.0,2.0,0.1,0.968142,0.871003
19,2.0,4.0,0.2,0.968142,0.871003
18,2.0,4.0,0.1,0.962832,0.869499
4,1.0,2.0,0.2,0.966372,0.869115
34,5.0,10.0,0.2,0.962832,0.868909
33,5.0,10.0,0.1,0.962832,0.868820
20,2.0,4.0,0.5,0.964602,0.866962
7,1.0,4.0,0.2,0.969912,0.865900
6,1.0,4.0,0.1,0.969912,0.864808


In [34]:
def text_search(query):
    boost_dict = {
        "question": 1.0,
        "answer": 2.0,
        "section": 0.1,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

# RAG Answer

In [35]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [36]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [37]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

# Running RAG

In [38]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [39]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
)

In [40]:
rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
answer_llm

'Yes — you can still join the course. If you want a certificate, make sure to submit your project while submissions are still being accepted.'

In [41]:
assistant.total_cost()

0.0005145

In [42]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [43]:
rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'I just found this course—can I still join it, or is it too late already?',
 'answer_llm': 'Yes — you can still join the course. If you want a certificate, make sure to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [44]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [45]:
answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': 'I just found this course—can I still join it, or is it too late already?',
 'answer_llm': 'Yes, you can still join. If you want a certificate, though, you need to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [46]:
assistant.reset_usage()

In [47]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [48]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/565 [00:00<?, ?it/s]

In [49]:
answers = []

for answer_record in results:
    answers.append(answer_record)

In [50]:
assistant.total_cost()

0.6130777500000004

In [51]:
df_answers = pd.DataFrame(answers)
df_answers.to_csv("data/rag-answers-new.csv", index=False)

# LLM as a Judge

In [52]:
import pandas as pd

df_answers = pd.read_csv("data/rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

In [53]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [54]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [55]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [56]:
from dotenv import load_dotenv
from openai import OpenAI
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

load_dotenv()
openai_client = OpenAI()

In [57]:
rec = answers[0]

In [58]:
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

In [59]:
eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the core meaning of the ground truth: it says you can still join, and that receiving a certificate requires submitting the project while submissions are still accepted. This is semantically equivalent.', score='good')

In [60]:
calc_price(usage)

{'input_cost': 0.00022199999999999998,
 'output_cost': 0.000252,
 'total_cost': 0.000474}

In [61]:
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [62]:
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the key meaning of the ground truth: the course can still be joined, and certificate eligibility depends on submitting the project before submissions close. This is semantically equivalent.', score='good')

In [63]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [64]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

  0%|          | 0/565 [00:00<?, ?it/s]

In [65]:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [66]:
df_eval = pd.DataFrame(evaluations)

In [67]:
calc_total_price(usages)

0.39932550000000017

In [68]:
good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

Good: 543/565 = 96.11%


In [69]:
df_eval[df_eval["score"] == "bad"].head()

,question,document,score,reasoning
4,Is the only requirement for the certificate to...,74eb249bbf,bad,The ground truth says that the only requiremen...
34,"If homework isn’t mandatory, what role does it...",9f689c185f,bad,The AI answer does not address the question or...
40,When is the next llm-zoomcamp course offering ...,bd31146b0e,bad,The ground truth states the next llm-zoomcamp ...
41,Do you know when this course will run again af...,bd31146b0e,bad,The ground truth gives a specific re-run time:...
102,Do I need to add money to my OpenAI account be...,152af39a53,bad,The AI answer partially matches the ground tru...


In [70]:
df_eval.to_csv("data/rag-evaluations-new.csv", index=False)